# TatHybrid - Phase 3C: Chain-of-Thought Prompt

**Optimization:** Phase 3C-3 - Chain-of-thought prompting (VALIDATION TEST)

**Dataset:** TatHybrid (Financial Reports - Numeracy Focused)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Numeracy F1 (numeracy-aware metric)

**Documents:** 4 example PDFs (inpixon, lifeway-foods, overseas-shipholding, viavi)

**Q&A Count:** 162 pairs (largest dataset)

**What changed:**
- ✅ Chain-of-thought prompt (step-by-step reasoning)
- ✅ Keep all Phase 2 parameters (TOP_K=10, CHUNK_SIZE=1500)

**Baseline (Phase 2):**
- Empty rate: 16.0% (26/162 questions)
- Numeracy F1: 57.91

**Target:**
- Empty rate: <12% (+6-8 questions)
- Numeracy F1: >60

**Expected runtime:** 50-70 minutes (162 Q&A, larger than FinHybrid)
**Expected cost:** 2x tokens (model generates reasoning + answer)

## Setup and Imports

In [1]:
import sys
import os

# Navigate to project root (4 levels up from 3_prompts/notebooks/)
project_root = os.path.abspath('../../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


In [2]:
import pandas as pd
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess
from uda.utils.prompts import get_prompt  # NEW: Import prompt module
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ All imports successful


## Configuration

In [3]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [4]:
# Experiment Parameters (SAME AS PHASE 2 - only prompt changes)
DATASET_NAME = "tat"
CHUNK_SIZE = 1500  # From Phase 2
CHUNK_OVERLAP = 150
TOP_K = 10  # From Phase 2
TEMPERATURE = 0.1
MAX_TOKENS = 512

# NEW: Prompt type
PROMPT_TYPE = "cot"  # instruction, fewshot, or cot

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/tathybrid_cot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Prompt type: {PROMPT_TYPE}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: tat
Chunk size: 1500
Top-K: 10
Prompt type: cot
Output dir: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/tathybrid_cot


## Initialize Models

In [5]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model (local, free)
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

# NEW: Load prompt function
prompt_fn = get_prompt(PROMPT_TYPE)
print(f"✓ Prompt function loaded: {PROMPT_TYPE}")

✓ Together AI client initialized


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Embedding model loaded: all-MiniLM-L6-v2
✓ Text splitter initialized
✓ Prompt function loaded: cot


## Helper Functions

In [6]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF using PyPDF2 (Phase 2 method)"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()
    
    # Delete if exists
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    
    # Create collection
    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )
    
    # Add documents
    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)
    
    return collection

def answer_question(collection, question):
    """
    Retrieve context and generate answer.
    
    CHANGED: Uses chain-of-thought prompt with step-by-step reasoning
    """
    # Retrieve (same as Phase 2)
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])
    
    # NEW: Build prompt using prompts module
    prompt_text = prompt_fn(context=context, question=question)
    
    # Convert to message format for Together AI
    messages = [
        {"role": "user", "content": prompt_text}
    ]
    
    # Generate (same as Phase 2)
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    
    return response.choices[0].message.content

print("✓ Helper functions defined")

✓ Helper functions defined


## Load Q&A Data

In [7]:
# Load TatHybrid Q&A
csv_file = "./dataset/qa/tat_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# Filter to only documents with available PDFs
AVAILABLE_DOCS = [
    "inpixon_2019",
    "lifeway-foods-inc_2019",
    "overseas-shipholding-group-inc_2019",
    "viavi-solutions-inc_2019"
]

qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")

Total documents in CSV: 170
Available PDFs: 4

Filtered to documents with PDFs:

  inpixon_2019: 18 Q&A pairs
  lifeway-foods-inc_2019: 60 Q&A pairs
  overseas-shipholding-group-inc_2019: 60 Q&A pairs
  viavi-solutions-inc_2019: 24 Q&A pairs

Total Q&A to process: 162


## Main Processing Loop

**This will process 4 documents with 162 Q&A pairs**

**Expected runtime:** 50-70 minutes (larger dataset than FinHybrid)

In [8]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")
    
    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue
    
    print(f"PDF: {pdf_path}")
    
    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")
    
    # Build index
    print("Building vector index...")
    collection = build_index(text_chunks, collection_name=f"tat_{doc_name}_cot")
    print("✓ Index built")
    
    # Process each question
    print(f"\nAnswering {len(doc_qas)} questions...")
    
    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")
        
        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")
            
            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
                "prompt_type": PROMPT_TYPE,
            })
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print(f"\n✓ Completed {doc_name}: {len([r for r in all_results if r['doc'] == doc_name])} questions processed")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")


Processing: overseas-shipholding-group-inc_2019
PDF: dataset/src_doc_files_example/tat_docs/overseas-shipholding-group-inc_2019.pdf
Extracting text...
Created 290 chunks
Building vector index...
✓ Index built

Answering 60 questions...

[1/60] What benefits are provided by the company to qualifying domestic retir...
   Answer: Based on the provided context, the Company provides the following benefits to qu...

[2/60] What is the change in Interest cost on benefit obligation for pension ...
   Answer: To answer this question, I need to find the "Interest cost on benefit obligation...

[3/60] What is the average Interest cost on benefit obligation for pension be...
   Answer: To find the average interest cost on benefit obligation for pension benefits for...

[4/60] In which year was Benefit obligation at beginning of year for pension ...
   Answer: Based on the "Change in benefit obligation" table in the context, the Benefit ob...

[5/60] What was the Interest cost on benefit obligatio

## Diagnostic: Check Empty Responses

In [9]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)
    
    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")
    
    # COMPARISON WITH PHASE 2
    phase2_empty = 26
    phase2_total = 162
    phase2_empty_pct = phase2_empty / phase2_total * 100
    
    improvement = phase2_empty - empty_count
    improvement_pct = phase2_empty_pct - (empty_count/total_count*100)
    
    print(f"\n{'='*80}")
    print(f"COMPARISON WITH PHASE 2 BASELINE")
    print(f"{'='*80}")
    print(f"Phase 2 (Baseline): {phase2_empty}/{phase2_total} empty ({phase2_empty_pct:.1f}%)")
    print(f"Phase 3C (CoT): {empty_count}/{total_count} empty ({empty_count/total_count*100:.1f}%)")
    print(f"\nImprovement: {improvement:+d} questions ({improvement_pct:+.1f} percentage points)")
    
    if improvement > 0:
        print(f"✅ SUCCESS: Chain-of-thought prompts reduced empty responses!")
    elif improvement == 0:
        print(f"⚠️  NEUTRAL: No change in empty responses")
    else:
        print(f"❌ REGRESSION: Empty responses increased")
    
    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
else:
    print("❌ No results to analyze")


DIAGNOSTIC: Empty Response Analysis
Total Q&A processed: 162
Empty responses: 29 (17.9%)
Answered: 133 (82.1%)

COMPARISON WITH PHASE 2 BASELINE
Phase 2 (Baseline): 26/162 empty (16.0%)
Phase 3C (CoT): 29/162 empty (17.9%)

Improvement: -3 questions (-1.9 percentage points)
❌ REGRESSION: Empty responses increased

Empty responses by document:
  overseas-shipholding-group-inc_2019: 14/60 empty (23.3%)
  lifeway-foods-inc_2019: 5/60 empty (8.3%)
  viavi-solutions-inc_2019: 5/24 empty (20.8%)
  inpixon_2019: 5/18 empty (27.8%)


## Evaluate Results

In [10]:
if all_results:
    print("\nEvaluating TatHybrid results (Numeracy F1)...")
    
    # WORKAROUND: Fix answer format for TatQA evaluation
    fixed_results = []
    for result in all_results:
        fixed_result = result.copy()
        answers = fixed_result['answers']
        if isinstance(answers, dict) and 'answer' in answers:
            if isinstance(answers['answer'], list) and len(answers['answer']) == 1:
                if answers.get('answer_type') in ['count', 'arithmetic']:
                    fixed_result['answers'] = {
                        'answer': answers['answer'][0],
                        'answer_type': answers['answer_type'],
                        'scale': answers['scale']
                    }
        fixed_results.append(fixed_result)
    
    eval_main(DATASET_NAME, fixed_results)
else:
    print("❌ No results to evaluate")


Evaluating TatHybrid results (Numeracy F1)...
Numerical F1 score: 8.80


## Save Results

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"tathybrid_cot_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")
    
    # Summary by document
    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/tathybrid_cot/tathybrid_cot_20260629_223459.csv
Total Q&A: 162

Results by document:
  overseas-shipholding-group-inc_2019: 60 questions
  lifeway-foods-inc_2019: 60 questions
  viavi-solutions-inc_2019: 24 questions
  inpixon_2019: 18 questions


## Final Summary

In [12]:
if all_results:
    results_df = pd.DataFrame(all_results)
    
    empty_count = results_df['response'].fillna('').str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count
    
    phase2_empty = 26
    improvement = phase2_empty - empty_count
    
    print(f"\n{'='*80}")
    print(f"FINAL SUMMARY - CHAIN-OF-THOUGHT PROMPT (TatHybrid)")
    print(f"{'='*80}")
    print(f"Dataset: TatHybrid (162 Q&A)")
    print(f"Prompt type: {PROMPT_TYPE}")
    print(f"\nResults:")
    print(f"  Answered: {answered_count}/{len(results_df)} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"  Empty: {empty_count}/{len(results_df)} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"\nVs Phase 2 Baseline:")
    print(f"  Change: {improvement:+d} questions")
    print(f"  Expected: +6 to +8 questions")
    print(f"  Cost: 2x tokens (reasoning + answer)")
    
    # Calculate if scaling is justified
    finHybrid_improvement = 4  # From FinHybrid test (+4 questions)
    finHybrid_total = 47
    finHybrid_pct = (finHybrid_improvement / finHybrid_total) * 100  # 8.5%
    
    tatHybrid_total = 162
    tatHybrid_pct = (improvement / tatHybrid_total) * 100
    
    print(f"\n{'='*80}")
    print(f"SCALING VALIDATION")
    print(f"{'='*80}")
    print(f"FinHybrid: +{finHybrid_improvement} questions (+{finHybrid_pct:.1f}%)")
    print(f"TatHybrid: +{improvement} questions (+{tatHybrid_pct:.1f}%)")
    
    if improvement >= 6:
        print(f"\n✅ EXCELLENT: CoT scales well to larger datasets!")
        print(f"   TatHybrid confirmed the FinHybrid results")
        print(f"   Recommendation: SCALE TO ALL 6 DATASETS")
    elif improvement >= 4:
        print(f"\n✅ GOOD: CoT works consistently across datasets")
        print(f"   Recommendation: SCALE TO ALL 6 DATASETS")
    elif improvement >= 2:
        print(f"\n⚠️  MODERATE: CoT helps but less than expected")
        print(f"   Recommendation: Scale but monitor closely")
    else:
        print(f"\n❌ FAILED: CoT did not scale to TatHybrid")
        print(f"   Recommendation: Investigate why, may not scale")
else:
    print("\n❌ No results to summarize")


FINAL SUMMARY - CHAIN-OF-THOUGHT PROMPT (TatHybrid)
Dataset: TatHybrid (162 Q&A)
Prompt type: cot

Results:
  Answered: 133/162 (82.1%)
  Empty: 29/162 (17.9%)

Vs Phase 2 Baseline:
  Change: -3 questions
  Expected: +6 to +8 questions
  Cost: 2x tokens (reasoning + answer)

SCALING VALIDATION
FinHybrid: +4 questions (+8.5%)
TatHybrid: +-3 questions (+-1.9%)

❌ FAILED: CoT did not scale to TatHybrid
   Recommendation: Investigate why, may not scale


---

## Done!

**Results saved to:** `./results/tathybrid_cot/`

**Validation Test Complete!**

If CoT scaled well (+6-8 questions), proceed to scale to all remaining datasets.
If not, investigate why and adjust strategy.